# Ticha HTR — wider beams and a right-to-left reranker for the end-to-end model

This notebook improves the decoding of the end-to-end model. It was first written for `trocr-base-handwritten-finetuned-smooth-cosine-aug` (`e2e-aug`): with 5 beams it reads the test lines without Za687 at CER 0.144, against 0.147 with 2 beams, and always picking the best of its 5 candidates would give 0.121. The system in the report is **run E** of the ablation notebook (`L2R_RUN = 'linear'`, the default) with 10 beams and the right-to-left reranker. Two ways to get closer to the best candidate:

1. **Wider beams (Part 1).** Decode validation and test with 2, 5, 8 and 10 beams; every candidate line is kept, with its log-probability, for Part 3.
2. **A right-to-left model (Part 2).** The same recipe as the best model (12 epochs, label smoothing 0.1, cosine schedule, lr 5e-5, augmented lines, best epoch by validation CER), trained to write each line's **word pieces in reverse order**. The pieces stay the ones the tokenizer makes left to right (`escri` + `tura`), only their order flips, so the pretrained vocabulary still fits and the model can score exactly the tokens of the left-to-right candidates.
3. **Reranking (Part 3).** Each candidate gets `((1 − w) · log p_L2R + w · log p_R2L) / tokens`, both computed from the image. Unlike a text-only language model (tried in an experiment not included in the repository), the right-to-left model sees the image, and it judges every token with the context on its right. `w` is chosen on validation, for the 5- and 10-beam candidates; `w = 0` is the left-to-right model alone.

Data: dataset **v1.1**. Its train split is the same as v1.0's, with Th687, so the right-to-left model saw the same training lines as the left-to-right model; its runs keep their folders (`...-r2l-th687`, `beams-r2l-rerank-th687`). Za687, the duplicate of Th687, is not in the v1.1 test split (936 lines after cleaning). The first run of this notebook (`beams-r2l-rerank`, `...-r2l`) dropped Th687 from train instead; only its left-to-right candidates are reused.

`L2R_RUN` picks the left-to-right model: `'cosine'` is `e2e-aug`; `'linear'` is run E of `ticha_trocr_ablation_smoothing_schedule.ipynb` (the same recipe with a linear schedule, which beat `e2e-aug` on validation), decoded from scratch into `beams-r2l-rerank-linear-aug-th687`. The right-to-left model is the same for both: it only scores candidates.

For `e2e-aug`, the left-to-right candidates do not depend on the right-to-left model, so the Th687 run copies them from `beams-r2l-rerank` (without the old right-to-left scores) instead of decoding again.

All models decode with start token `0`, the one they were trained with (set in `generation_config` too; the checkpoint saves `2`). On 150 validation lines both gave the same CER, but the numbers here can differ slightly from earlier experiments that used `2`.

**Before running:** train `e2e-aug` with [`ticha_trocr_finetuning.ipynb`](ticha_trocr_finetuning.ipynb) or run E with [`ticha_trocr_ablation_smoothing_schedule.ipynb`](ticha_trocr_ablation_smoothing_schedule.ipynb); both are read from `$TICHA_RUNS_DIR` (default `~/ticha-runs`). The two models are published together in [`maclenn77/ticha-trocr-base`](https://huggingface.co/maclenn77/ticha-trocr-base) and can be used without training, see [`ticha_trocr_usage_example.ipynb`](ticha_trocr_usage_example.ipynb).

**Run on the VM** (`ticha-l4`, kernel **Python (ticha-trocr)**). Part 2 takes about 3.5 hours; everything is saved to `~/ticha-runs/` and reloaded on a rerun (candidates, the right-to-left model, its scores).

In [ ]:
import os
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import copy
import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import concatenate_datasets, load_dataset
from IPython.display import display
from jiwer import cer, process_characters, wer
from transformers import (EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrOCRProcessor,
                          VisionEncoderDecoderModel)
from transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder import shift_tokens_right

RUNS = Path(os.environ.get('TICHA_RUNS_DIR', '~/ticha-runs')).expanduser()
# v1.1: train as in v1.0, with Th687, like the left-to-right model; Za687, its duplicate, is not in test.
DATASET_REVISION = 'v1.1'
SUFFIX = '-th687'  # the run folders of the models trained with Th687
R2L_DIR = RUNS / f'trocr-base-handwritten-finetuned-smooth-cosine-aug-r2l{SUFFIX}'
# The left-to-right model whose beams are reranked: 'cosine' = e2e-aug (smoothing, cosine + augmentation), the
# first run of this notebook; 'linear' = run E of the ablation notebook (smoothing, linear + augmentation).
L2R_RUN = 'linear'
L2R_DIR = {'cosine': RUNS / 'trocr-base-handwritten-finetuned-smooth-cosine-aug',
           'linear': RUNS / 'ablation-smoothing-schedule' / 'e-smoothing-linear-augmentation'}[L2R_RUN]
OUT = RUNS / (f'beams-r2l-rerank{SUFFIX}' if L2R_RUN == 'cosine' else f'beams-r2l-rerank-linear-aug{SUFFIX}')
# Left-to-right candidates already decoded by the first run (only valid for the e2e-aug model)
L2R_CACHE = RUNS / 'beams-r2l-rerank' if L2R_RUN == 'cosine' else OUT
SMOKE = os.environ.get('TICHA_SMOKE') == '1'  # quick end-to-end check: few lines, a few training steps, scratch folders
if SMOKE:
    R2L_DIR, OUT = RUNS / 'smoke-r2l' / 'model', RUNS / 'smoke-r2l'
OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'microsoft/trocr-base-handwritten'

if not torch.cuda.is_available():
    raise RuntimeError('No GPU: run this notebook on the VM.')
DEVICE = torch.device('cuda')
USE_BF16 = torch.cuda.is_bf16_supported()
BEAMS = [2, 5, 8, 10]
RERANK_BEAMS = [5, 10]
RERANK_WEIGHTS = [round(w, 2) for w in np.arange(0, 1.01, 0.1)]
DECODE_BATCH = 8          # lines per generate call (x beams sequences)
SCORE_BATCH = 32          # sequences per teacher-forced scoring batch
START = 0                 # decoder start token the models were trained with
SEED = 42
print('GPU:', torch.cuda.get_device_name(0), '| bf16:', USE_BF16, '| outputs:', OUT)

## Data

The cleanup of the training notebooks (`clean_text`): abbreviation expansions and glosses removed, lines with damage or illegible text dropped, uncertain readings kept as text in train and dropped from validation and test. Train documents that mostly repeat validation or test lines are listed as a check; v1.1 has none.

In [ ]:
BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',
    re.IGNORECASE)
DROP_HASH = True
GUESS_RE = re.compile(r'\((?!word\?)([^()?]*[^\W\d_][^()?]*)\?+\)')


# Same rule as ticha.core.htr.clean_text, written out so the notebook runs without the package.
def clean_text(text, keep_guesses=False):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    if keep_guesses:
        text = GUESS_RE.sub(lambda m: m.group(1), text)
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or (DROP_HASH and '#' in text):
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


def clean_split(split, keep_guesses):
    # Clean the text column in memory and keep lines by index: map/filter/cast would write a full copy of the
    # images to the datasets cache every run (that filled the VM's disk once).
    before = len(split)
    cleaned = [clean_text(t, keep_guesses) for t in split['text']]
    split = split.remove_columns('text').add_column('text', cleaned)
    return split.select([i for i, t in enumerate(cleaned) if t is not None]), before


ds = load_dataset('maclenn77/ticha-htr-lines', 'original', revision=DATASET_REVISION)
for name in ds:
    ds[name], before = clean_split(ds[name], keep_guesses=name == 'train')
    print(f'{name}: kept {len(ds[name]):,} / {before:,} lines')


def line_key(text):
    return re.sub(r'\W+', ' ', text.lower()).strip()


eval_keys = {line_key(t) for name in ['validation', 'test'] for t in ds[name]['text'] if len(t) >= 15}
by_doc = {}
for doc_id, text in zip(ds['train']['doc_id'], ds['train']['text']):
    by_doc.setdefault(doc_id, []).append(line_key(text) in eval_keys)
duplicates = {doc for doc, hits in by_doc.items() if np.mean(hits) > 0.5}
print('Train documents that duplicate evaluation documents:', sorted(duplicates) or 'none')

aug, before = clean_split(load_dataset('maclenn77/ticha-htr-lines', 'augmented', revision=DATASET_REVISION)['train'], keep_guesses=True)
shared = ['image', 'text', 'doc_id']  # all the training needs
aug = aug.select_columns(shared)
if aug.features != ds['train'].select_columns(shared).features:
    aug = aug.cast(ds['train'].select_columns(shared).features)
train_aug = concatenate_datasets([ds['train'].select_columns(shared), aug]).shuffle(seed=SEED)
print(f'augmented: kept {len(aug):,} / {before:,} lines -> {len(train_aug):,} training lines')

if SMOKE:
    ds['validation'], ds['test'] = ds['validation'].select(range(16)), ds['test'].select(range(16))
    train_aug = train_aug.select(range(64))
meta = {name: ds[name].remove_columns('image').to_pandas() for name in ['validation', 'test']}

## Part 1 — Wider beams

The left-to-right model decodes every validation and test line with each beam width in `BEAMS`, returning all its candidates. Each candidate's log-probability is then recomputed by teacher forcing (the sum over its tokens up to `</s>`), so all widths and both directions are scored the same way. The first candidate is beam search's own choice. Candidates are cached in `OUT`.

In [ ]:
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
BOS, PAD, EOS = tokenizer.cls_token_id, tokenizer.pad_token_id, tokenizer.sep_token_id
token_lengths = [len(tokenizer(t).input_ids) for t in ds['train']['text']]
MAX_LENGTH = min(256, max(64, int(np.percentile(token_lengths, 99)) + 8))
print('Max target length:', MAX_LENGTH)


def configure(m):
    for cfg in (m.config, m.generation_config):
        cfg.decoder_start_token_id, cfg.pad_token_id, cfg.eos_token_id = START, PAD, EOS
    m.config.vocab_size = m.config.decoder.vocab_size
    m.generation_config.max_length = MAX_LENGTH
    m.generation_config.early_stopping = True
    return m


def load_model(path):
    return configure(VisionEncoderDecoderModel.from_pretrained(path)).to(DEVICE).eval()


def pixel_batches(split, size):
    for start in range(0, len(split), size):
        rows = split.select(range(start, min(start + size, len(split))))
        images = [image.convert('RGB') for image in rows['image']]
        yield start, processor(images=images, return_tensors='pt').pixel_values.to(DEVICE)


def inner_tokens(ids):
    """The tokens between the start/<s> tokens and the first </s> of a generated or labelled sequence."""
    ids = [int(i) for i in ids]
    while ids and ids[0] in (START, BOS):
        ids = ids[1:]
    return ids[:ids.index(EOS)] if EOS in ids else [i for i in ids if i != PAD]


def target(inner, direction):
    """Labels as in training: <s>, the tokens (reversed for r2l), </s>."""
    return [BOS] + (inner[::-1] if direction == 'r2l' else inner) + [EOS]


@torch.inference_mode()
def sequence_log_probs(model, pixel_values, label_lists):
    """Teacher-forced summed log-probability of each label sequence; one image row per sequence."""
    width = max(map(len, label_lists))
    labels = torch.full((len(label_lists), width), -100, dtype=torch.long)
    for row, ids in enumerate(label_lists):
        labels[row, :len(ids)] = torch.tensor(ids)
    labels = labels.to(DEVICE)
    decoder_inputs = shift_tokens_right(labels, PAD, START)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
        logits = model(pixel_values=pixel_values, decoder_input_ids=decoder_inputs).logits
    logp = logits.float().log_softmax(-1).gather(-1, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    return (logp * (labels != -100)).sum(-1).tolist()


@torch.inference_mode()
def nbest(model, split_name, beams):
    path = OUT / f'nbest_{split_name}_beam{beams}.json'
    if path.exists():
        return json.loads(path.read_text())
    cached = L2R_CACHE / path.name
    if cached != path and cached.exists() and not SMOKE:
        rows = json.loads(cached.read_text())
        for row in rows:
            for c in row['candidates']:
                c.pop('r2l', None)  # scores of the other right-to-left model
        path.write_text(json.dumps(rows))
        print(f'{split_name}, {beams} beams: left-to-right candidates copied from {L2R_CACHE}', flush=True)
        return rows
    split, rows, t0 = ds[split_name], [], time.time()
    ids = split['id']
    for start, pixel_values in pixel_batches(split, DECODE_BATCH):
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            sequences = model.generate(pixel_values=pixel_values, num_beams=beams, num_return_sequences=beams,
                                       max_length=MAX_LENGTH, early_stopping=True)
        inners = [inner_tokens(seq) for seq in sequences.tolist()]
        repeated = pixel_values.repeat_interleave(beams, 0)
        scores = sequence_log_probs(model, repeated, [target(i, 'l2r') for i in inners])
        for line in range(pixel_values.size(0)):
            cands = []
            for k in range(beams):
                inner = inners[line * beams + k]
                cands.append({'text': tokenizer.decode(inner, skip_special_tokens=True).strip(), 'inner': inner,
                              'l2r': scores[line * beams + k], 'tokens': len(inner) + 2})
            rows.append({'id': ids[start + line], 'candidates': cands})
    print(f'{split_name}, {beams} beams: {(time.time() - t0) / len(split):.2f} s/line', flush=True)
    path.write_text(json.dumps(rows))
    return rows


l2r = load_model(L2R_DIR)
candidates = {(split, b): nbest(l2r, split, b) for b in BEAMS for split in ['validation', 'test']}

In [ ]:
references = {split: dict(zip(meta[split]['id'], meta[split]['text'])) for split in meta}
languages = {split: dict(zip(meta[split]['id'], meta[split]['language'])) for split in meta}
# Only the lines of the loaded splits: candidates saved before v1.1 can include Za687.
keep = {split: set(ids).__contains__ for split, ids in references.items()}


def evaluate(split, predictions):
    """CER/WER of {id: text} on a split, overall and by language."""
    ids = [i for i in predictions if keep[split](i)]
    refs, preds = [references[split][i] for i in ids], [predictions[i] or ' ' for i in ids]
    row = {'lines': len(ids), 'CER': cer(refs, preds), 'WER': wer(refs, preds)}
    for lang in sorted({languages[split][i] for i in ids}):
        idx = [n for n, i in enumerate(ids) if languages[split][i] == lang]
        row[f'CER {lang}'] = cer([refs[n] for n in idx], [preds[n] for n in idx])
    return row


def char_errors(split, predictions):
    ids = sorted(i for i in predictions if keep[split](i))
    errors, lengths = [], []
    for i in ids:
        o = process_characters(references[split][i], predictions[i] or ' ')
        errors.append(o.substitutions + o.deletions + o.insertions)
        lengths.append(len(references[split][i]))
    return ids, np.array(errors), np.array(lengths)


def paired_bootstrap(split, base, new, n=5000):
    """CER difference new - base, with a 95% interval from resampling whole documents."""
    ids, e_base, lengths = char_errors(split, base)
    _, e_new, _ = char_errors(split, new)
    docs = np.array([i.split('_')[0] for i in ids])
    groups = [np.where(docs == d)[0] for d in np.unique(docs)]
    rng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(n):
        s = np.concatenate([groups[g] for g in rng.integers(len(groups), size=len(groups))])
        diffs.append((e_new[s].sum() - e_base[s].sum()) / lengths[s].sum())
    diffs = np.array(diffs)
    return {'difference': (e_new.sum() - e_base.sum()) / lengths.sum(), 'low': np.percentile(diffs, 2.5),
            'high': np.percentile(diffs, 97.5), 'share better': float(np.mean(diffs < 0))}


def first_choice(rows):
    return {row['id']: row['candidates'][0]['text'] for row in rows}


def oracle(split, rows):
    return {row['id']: min(row['candidates'], key=lambda c: cer(references[split][row['id']], c['text'] or ' '))['text']
            for row in rows}


beam_table = pd.DataFrame({
    f'{b} beams': {**{f'validation {k}': v for k, v in evaluate('validation', first_choice(candidates[('validation', b)])).items()
                      if k in ('CER', 'WER')},
                   **{f'test {k}': v for k, v in evaluate('test', first_choice(candidates[('test', b)])).items() if k != 'lines'},
                   'validation oracle CER': evaluate('validation', oracle('validation', candidates[('validation', b)]))['CER'],
                   'test oracle CER': evaluate('test', oracle('test', candidates[('test', b)]))['CER']}
    for b in BEAMS}).T
print('Left-to-right model by beam width (test without Za687; oracle = best candidate always picked)')
display(beam_table.round(4))
beam_table.to_csv(OUT / 'beam_sweep.csv')

for b in BEAMS[1:]:
    for split in ['validation', 'test']:
        result = paired_bootstrap(split, first_choice(candidates[(split, 2)]), first_choice(candidates[(split, b)]))
        print(f"{b} beams vs 2, {split}: {result['difference']:+.4f} CER "
              f"[{result['low']:+.4f}, {result['high']:+.4f}], better in {result['share better']:.1%} of resamples")

## Part 2 — Train the right-to-left model

The recipe of `smooth-cosine-aug` in `ticha_trocr_finetuning.ipynb`, unchanged except the direction: the labels are `<s>`, the line's word pieces in reverse order, `</s>`. Validation CER during training decodes the reversed pieces and flips them back. The best epoch is kept (early stopping, patience 2). If `R2L_DIR` already holds a trained model, it is loaded instead.

In [ ]:
class ReversedCollator:
    """Image batches whose labels are the word pieces of each line in reverse order."""

    def __call__(self, batch):
        images = [item['image'].convert('RGB') for item in batch]
        pixel_values = processor(images=images, return_tensors='pt').pixel_values
        label_lists = [target(tokenizer(str(item['text']).strip(), add_special_tokens=False).input_ids[:MAX_LENGTH - 2], 'r2l')
                       for item in batch]
        labels = torch.full((len(batch), MAX_LENGTH), -100, dtype=torch.long)
        for row, ids in enumerate(label_lists):
            labels[row, :len(ids)] = torch.tensor(ids)
        # With label smoothing the Trainer removes `labels` before the forward pass, so pass the decoder inputs too.
        return {'pixel_values': pixel_values, 'labels': labels,
                'decoder_input_ids': shift_tokens_right(labels, PAD, START)}


def r2l_text(ids):
    return tokenizer.decode(inner_tokens(ids)[::-1], skip_special_tokens=True).strip()


def r2l_metrics(eval_pred):
    preds = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    labels = np.where(eval_pred.label_ids == -100, PAD, eval_pred.label_ids)
    pred_text = [r2l_text(p) for p in preds]
    ref_text = [r2l_text(l) for l in labels]
    return {'cer': cer(ref_text, [p or ' ' for p in pred_text]), 'wer': wer(ref_text, [p or ' ' for p in pred_text])}


# Check the reversal on a few lines before training.
for text in ds['validation']['text'][:3]:
    inner = tokenizer(text, add_special_tokens=False).input_ids
    assert r2l_text(target(inner, 'r2l')) == text.strip(), text
    print(repr(text), '->', tokenizer.convert_ids_to_tokens(target(inner, 'r2l'))[:8], '...')

if (R2L_DIR / 'model.safetensors').exists():
    print('Using the saved right-to-left model in', R2L_DIR)
else:
    del l2r
    gc.collect()
    torch.cuda.empty_cache()
    model = configure(VisionEncoderDecoderModel.from_pretrained(MODEL_ID))
    model.generation_config.num_beams = 2
    args = Seq2SeqTrainingArguments(
        output_dir=str(R2L_DIR), per_device_train_batch_size=8, per_device_eval_batch_size=8,
        gradient_accumulation_steps=1, optim='adamw_torch', learning_rate=5e-5, weight_decay=0.01,
        lr_scheduler_type='cosine', warmup_ratio=0.05, label_smoothing_factor=0.1, num_train_epochs=12,
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='steps', logging_steps=100,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='cer', greater_is_better=False,
        predict_with_generate=True, generation_max_length=MAX_LENGTH, generation_num_beams=2,
        bf16=USE_BF16, fp16=not USE_BF16, remove_unused_columns=False, dataloader_num_workers=4,
        report_to='none', seed=SEED, **({'max_steps': 10} if SMOKE else {}))
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_aug, eval_dataset=ds['validation'],
                             data_collator=ReversedCollator(), compute_metrics=r2l_metrics,
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    t0 = time.time()
    trainer.train()
    trainer.save_model(str(R2L_DIR))
    processor.save_pretrained(str(R2L_DIR))
    trainer.save_state()
    print(f'Trained in {(time.time() - t0) / 3600:.1f} h')
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

history = json.loads((R2L_DIR / 'trainer_state.json').read_text())
r2l_curve = pd.DataFrame([{'epoch': round(h['epoch']), 'validation CER': h['eval_cer'], 'validation WER': h['eval_wer'],
                           'validation loss': h['eval_loss']} for h in history['log_history'] if 'eval_cer' in h]).set_index('epoch')
print('Right-to-left model, validation per epoch (greedy-ish decoding, 2 beams, reversed back):')
display(r2l_curve.round(4))
print('Best validation CER:', round(history['best_metric'], 4), '| left-to-right model with the same decoding: 0.165')

## Part 3 — Rerank the candidates with the right-to-left model

Each left-to-right candidate is scored by the right-to-left model, from the same image, on its word pieces in reverse order. `w` is chosen on validation for each beam width; the test split is then scored once.

In [ ]:
r2l = load_model(R2L_DIR)


@torch.inference_mode()
def add_r2l_scores(split_name, beams):
    path = OUT / f'nbest_{split_name}_beam{beams}.json'
    rows = candidates[(split_name, beams)]
    if all('r2l' in c for row in rows for c in row['candidates']):
        return
    split = ds[split_name]
    ids = [row['id'] for row in rows]
    if split['id'] != ids:  # cached candidates can cover fewer lines (test caches were saved without Za687)
        position = {line_id: n for n, line_id in enumerate(split['id'])}
        split = split.select([position[line_id] for line_id in ids])
    lines_per_batch = max(1, SCORE_BATCH // beams)
    for start, pixel_values in pixel_batches(split, lines_per_batch):
        chunk = rows[start:start + pixel_values.size(0)]
        cands = [c for row in chunk for c in row['candidates']]
        repeated = pixel_values.repeat_interleave(beams, 0)
        for c, score in zip(cands, sequence_log_probs(r2l, repeated, [target(c['inner'], 'r2l') for c in cands])):
            c['r2l'] = score
    path.write_text(json.dumps(rows))
    print(f'Scored {split_name}, {beams} beams, with the right-to-left model', flush=True)


for b in RERANK_BEAMS:
    for split in ['validation', 'test']:
        add_r2l_scores(split, b)


def rerank(rows, w):
    return {row['id']: max(row['candidates'], key=lambda c: ((1 - w) * c['l2r'] + w * c['r2l']) / c['tokens'])['text']
            for row in rows}


sweep = pd.DataFrame({f'{b} beams': {w: evaluate('validation', rerank(candidates[('validation', b)], w))['CER']
                                     for w in RERANK_WEIGHTS} for b in RERANK_BEAMS})
sweep.index.name = 'w'
print('Validation CER by right-to-left weight w (w = 0: left-to-right scores only; w = 1: right-to-left only)')
display(sweep.round(4))
best_w = {b: float(sweep[f'{b} beams'].idxmin()) for b in RERANK_BEAMS}
print('Chosen on validation:', best_w)

## Test results

Every configuration with its validation-chosen `w`, against the left-to-right model with 2 and 5 beams. Test leaves Za687 out. The difference against 5 beams alone gets a 95% interval from resampling whole documents.

In [ ]:
systems = {'L2R, 2 beams': ('test', first_choice(candidates[('test', 2)])),
           'L2R, 5 beams': ('test', first_choice(candidates[('test', 5)]))}
for b in BEAMS[2:]:
    systems[f'L2R, {b} beams'] = ('test', first_choice(candidates[('test', b)]))
for b in RERANK_BEAMS:
    systems[f'L2R {b} beams + R2L rerank (w={best_w[b]})'] = ('test', rerank(candidates[('test', b)], best_w[b]))
    systems[f'oracle, best of {b}'] = ('test', oracle('test', candidates[('test', b)]))

final = pd.DataFrame({name: evaluate(split, preds) for name, (split, preds) in systems.items()}).T
print('Test without Za687 (936 lines)')
display(final.round(4))
final.to_csv(OUT / 'test_results_without_Za687.csv')

base = systems['L2R, 5 beams'][1]
for name, (split, preds) in systems.items():
    if 'rerank' in name or name.startswith('L2R, 8') or name.startswith('L2R, 10'):
        r = paired_bootstrap('test', base, preds)
        print(f"{name} vs 5 beams: {r['difference']:+.4f} CER [{r['low']:+.4f}, {r['high']:+.4f}], "
              f"better in {r['share better']:.1%} of resamples")

pd.DataFrame({'id': list(base), 'reference': [references['test'][i] for i in base],
              **{name: [preds[i] for i in base] for name, (split, preds) in systems.items()}}).to_csv(
    OUT / 'test_predictions.csv', index=False)

In [ ]:
# Validation lines the reranker changed (5 beams), to see what the right-to-left model fixes and breaks.
b = RERANK_BEAMS[0]
rows = candidates[('validation', b)]
alone, reranked = first_choice(rows), rerank(rows, best_w[b])
changed = pd.DataFrame([{'reference': references['validation'][i], 'L2R': alone[i], 'reranked': reranked[i],
                         'CER change': cer(references['validation'][i], reranked[i] or ' ') - cer(references['validation'][i], alone[i] or ' ')}
                        for i in alone if alone[i] != reranked[i]])
if len(changed):
    print(f'{len(changed)} of {len(rows)} validation lines changed: {(changed["CER change"] < 0).sum()} better, '
          f'{(changed["CER change"] > 0).sum()} worse')
    pd.set_option('display.max_colwidth', 80)
    display(changed.sort_values('CER change').head(10))
    display(changed.sort_values('CER change').tail(10))
else:
    print('The chosen w changes no validation line.')

## How to read the results

- **Beams:** a wider beam only helps if the extra candidates contain better lines *and* the model ranks them first. The oracle column shows how much better lines the extra candidates contain.
- **Reranker:** compare the reranked rows with the left-to-right row of the same width. A gain means the right-to-left model, which reads the same image with the opposite context, can tell good candidates from bad ones where the left-to-right model alone could not.
- `w` was chosen on validation; only the test rows of the chosen settings are fair results.